# CartPole 值学习 · v1 表格 Q-learning（故意不用神经网络）

**这个 notebook 在做什么**：在 CartPole（小车顶一根杆，动作是把小车往左/右推）上训一个最朴素的表格 Q-learning，让小车学会不断调整位置把杆顶住不倒。

**为什么从表格 Q 开始，而且故意不用神经网络**：值学习最原始的形式就是一张查找表——每个状态、每个动作对应一个数字 Q(s,a)，训练就是不断把这个数字往真实回报上修正。CartPole 的观测是 4 维连续量，表格方法要求状态是离散索引，所以第一步永远是**分桶**：把每一维切成有限段。分桶越细 Q 表越准，但表的大小是桶数的指数级乘积——这一级刻意把"不用网络会撞到什么墙"这件事留给你亲眼看到：桶粒度是个死板的超参，调参空间很快就见顶。这正是下一级 `train_v2_dqn.py` 引入神经网络（拿泛化换精确）要解决的问题。

**与前后模块的关系**：本模块（`3_1_cartpole_value_rl`）是组 3（off-policy）值学习的地基，讲清"表格 → 网络 → 回放 → 目标网络"这条主线；`3_2_so101_offpolicy/` 会在 SO101 连续控制任务上把值学习升级成 DDPG → TD3 → SAC。

> **运行方式**：CartPole 纯 CPU 即可训练，无需 GPU。自上而下逐 cell 运行；训完 Q 表存到 `DATASETS_ROOT/models/trained/cartpole/qtable.npy`。两级都打印回合回报的滑动均值，直接看打印结果对照即可，无需额外 rollout 脚本。

In [ ]:
"""表格 Q-learning：CartPole 值学习的第一级，故意不用神经网络、不用 Lightning。

CartPole 的观测是 4 维连续量（小车位置、小车速度、杆的角度、杆的角速度），而表格方法
要求状态是离散索引，所以第一步永远是**分桶**：把每一维切成有限段，四维分桶组合起来
就是 Q 表的“状态”维度。分桶越细、Q 表越准，但表的大小是每维桶数的**指数级乘积**——
只有 4 维观测就已经要精心挑选桶数和裁剪范围才能学好，换成图像或更高维的机器人状态就
直接组合爆炸、根本存不下。这正是下一级 `train_v2_dqn.py` 用神经网络代替查表
（拿泛化换精确、用函数近似绕开维度诅咒）要解决的问题。

更新规则是最朴素的表格 Q-learning（TD(0)、off-policy）：
    Q[s, a] += ALPHA * (r + GAMMA * max_a' Q[s', a'] - Q[s, a])
行为策略是 ε-greedy，ε 随回合数线性衰减，训练前期多探索、后期多利用当前 Q 表。
"""
from __future__ import annotations

import os
from collections import deque
from pathlib import Path

import gymnasium as gym
import numpy as np

## 1 状态分桶：把连续观测切成离散索引

`NUM_BINS` 是每一维切成的桶数，四维状态空间大小就是 `NUM_BINS ** 4`——这已经是表格方法能承受的极限量级。`OBS_LOW/OBS_HIGH` 是每一维参与分桶的裁剪范围：位置、角度按物理终止边界给，速度、角速度理论上无界，所以用比理论范围更紧的经验区间——真正学到东西的样本大多落在这个区间里，切太宽反而把桶都浪费在几乎不出现的极端速度上。`discretize` 用 `np.digitize` 把一个 4 维观测映射成一个桶索引 tuple，直接当 Q 表的行索引用。

In [ ]:
# 每一维切成的桶数：状态空间大小 = NUM_BINS ** 4，四维已经是表格方法的极限量级。
NUM_BINS = 8
# 每一维参与分桶的裁剪范围（超出范围的观测会被夹到边界桶）。位置/角度用物理终止边界，
# 速度/角速度理论上无界，用比理论范围更紧的经验区间——真正学到东西的样本大多落在这个
# 区间里，切得太宽会把桶都浪费在几乎不出现的极端速度上。
OBS_LOW = np.array([-2.4, -2.0, -0.21, -2.5])
OBS_HIGH = np.array([2.4, 2.0, 0.21, 2.5])
BIN_EDGES = [np.linspace(low, high, NUM_BINS - 1) for low, high in zip(OBS_LOW, OBS_HIGH)]


def discretize(obs):
    """把 4 维连续观测映射成离散桶索引 tuple，作为 Q 表的行索引。"""
    return tuple(int(np.digitize(value, edges)) for value, edges in zip(obs, BIN_EDGES))

## 2 表格 Q 与更新超参

`ALPHA`（学习率）、`GAMMA`（折扣因子）是 TD 更新公式里的两个系数；`EPSILON_*` 三个常量控制 ε-greedy 的探索强度随回合数从 1.0 线性衰减到 0.02——前期几乎纯随机探索地图，后期基本利用当前学到的 Q 表。`EPISODES` 是总训练回合数，`PRINT_EVERY` 定义了统一的统计口径：每 N 回合打印一次最近 N 回合回报的滑动均值，v2 DQN 用同样的格式打印，方便直接对照两级的学习曲线。

In [ ]:
ALPHA = 0.2  # 学习率
GAMMA = 0.99  # 折扣因子

EPSILON_START = 1.0
EPSILON_END = 0.02
EPSILON_DECAY_EPISODES = 2500  # 到第几回合线性衰减到 EPSILON_END，之后保持不变

EPISODES = 4000
PRINT_EVERY = 200  # 每 N 回合打印一次「最近 N 回合回报的滑动均值」——两级统一的统计口径

## 3 训练循环：ε-greedy 采样 + TD 更新

`main` 里每一步的核心只有两行：ε-greedy 选动作，然后用表格 Q-learning 的 TD 更新公式修正 `Q[state][action]`。注意 `target` 的分支——真正的失败（`terminated`，杆倒/出界）不 bootstrap 下一状态的价值，因为已经没有下一状态了；而到时间上限的 `truncated` 只是环境计时器切断，本该继续的轨迹被强行截断，所以仍要把下一状态的估计接上去，否则会把“还活得好好的”状态错误地当成回报为 0 的终止态。跑完 `EPISODES` 回合后把整张 Q 表存盘，供后续需要时直接加载。

In [ ]:
def main():
    env = gym.make("CartPole-v1")
    n_actions = env.action_space.n
    Q = np.zeros((NUM_BINS,) * 4 + (n_actions,))

    recent_returns = deque(maxlen=PRINT_EVERY)
    for episode in range(1, EPISODES + 1):
        epsilon = max(
            EPSILON_END,
            EPSILON_START - (EPSILON_START - EPSILON_END) * episode / EPSILON_DECAY_EPISODES,
        )
        obs, _ = env.reset()
        state = discretize(obs)
        episode_return = 0.0
        done = False
        while not done:
            if np.random.rand() < epsilon:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            next_state = discretize(next_obs)

            # 真正的失败（杆倒/出界）不 bootstrap 下一状态；到时间上限的 truncated 只是
            # 环境计时器切断，不代表回报到此为止，仍要接上下一状态的估计。
            target = reward if terminated else reward + GAMMA * np.max(Q[next_state])
            Q[state][action] += ALPHA * (target - Q[state][action])

            state = next_state
            episode_return += reward

        recent_returns.append(episode_return)
        if episode % PRINT_EVERY == 0:
            print(
                f"episode {episode:5d} | epsilon {epsilon:.3f} | "
                f"return (avg over last {PRINT_EVERY}) {np.mean(recent_returns):6.1f}"
            )

    qtable_path = Path(os.environ["DATASETS_ROOT"]) / "models" / "trained" / "cartpole" / "qtable.npy"
    qtable_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(qtable_path, Q)
    print(f"Q 表已保存到 {qtable_path}")
    env.close()

In [ ]:
if __name__ == "__main__":
    main()